# Model 1: SentenceTransformer + Regression DNN

**Architecture:** `all-MiniLM-L6-v2` (frozen, 22M params) → 384-dim dense embedding → DNN head (1024-dim, 6 ResidualBlocks, ~13M trainable params)

**Target:** Beat BoW DNN baseline (MAE $46.49)

**Key technique:** Pre-compute embeddings once → train regression head only → fast training

## vast.ai Setup (chỉ chạy lần đầu khi thuê máy)

Sau khi `git clone` repo và `cd` vào đúng thư mục, mở terminal trên vast.ai và chạy:

```bash
pip install uv
uv sync
```

Sau đó khởi động lại Jupyter kernel rồi chạy các cell bên dưới.

In [1]:
from pricer.items import Item
from pricer.sentence_transformer_model import SentTransRunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [2]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 800,000 | Val: 10,000 | Test: 10,000


## 2. Setup Model

Pre-computing 800k embeddings with SentenceTransformer. This is the most time-consuming step (~10-15 min on GPU).

In [3]:
runner = SentTransRunner(train, val[:1000])
runner.setup()

Loading SentenceTransformer encoder (frozen)...
Pre-computing train embeddings...


Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

Pre-computing val embeddings...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

SentTrans DNN created with 203,063,297 trainable parameters
Using cuda


## 3. Train

Max 15 epochs with early stopping (patience=3). CosineAnnealingLR schedule.

In [4]:
history = runner.train(epochs=15, patience=3)

Epoch 1/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [1/15]
  Train Loss: 0.7169, Val Loss: 0.5087
  Val MAE: $71.44, LR: 0.001000
  ** New best Val MAE: $71.44


Epoch 2/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [2/15]
  Train Loss: 0.4712, Val Loss: 0.4768
  Val MAE: $66.27, LR: 0.000989
  ** New best Val MAE: $66.27


Epoch 3/15:   0%|          | 0/3125 [00:00<?, ?it/s]

...

Epoch [13/15]
  Train Loss: 0.2649, Val Loss: 0.3826
  Val MAE: $50.18, LR: 0.000095
  No improvement (1/3)


Epoch 14/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [14/15]
  Train Loss: 0.2569, Val Loss: 0.3842
  Val MAE: $50.30, LR: 0.000043
  No improvement (2/3)


Epoch 15/15:   0%|          | 0/3125 [00:00<?, ?it/s]

Epoch [15/15]
  Train Loss: 0.2526, Val Loss: 0.3826
  Val MAE: $49.99, LR: 0.000011
  ** New best Val MAE: $49.99


## 4. Training History

In [5]:
plot_training_history(history, title="SentenceTransformer + DNN")

## 5. Evaluate on 200 Test Samples

Using `evaluate()` from `pricer/evaluator.py` — same evaluation framework as all other models.

In [6]:
evaluate(runner.inference, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$86 $36 $29 $21 $61 $55 $36 $6 $2 $83 $102 $166 $3 $10 $15 $6 $1 $1 $18 $20 $20 $10 $20 $93 $99 $284 $136 $9 $111 $50 $32 $25 $56 $35 $8 $278 $52 $18 $171 $5 $25 $49 $9 $109 $41 $11 $1 $4 $97 $5 $11 $38 $45 $14 $13 $61 $16 $72 $91 $16 $127 $43 $16 $44 $295 $10 $16 $197 $41 $40 $19 $5 $78 $25 $8 $14 $111 $1 $5 $9 $38 $20 $2 $64 $2 $36 $91 $3 $28 $39 $41 $6 $3 $4 $2 $43 $3 $22 $42 $140 $40 $19 $3 $82 $2 $33 $10 $237 $2 $172 $27 $47 $1 $43 $32 $49 $10 $31 $64 $20 $19 $0 $27 $26 $42 $17 $0 $37 $31 $67 $17 $81 $18 $2 $39 $11 $47 $8 $54 $24 $55 $110 $10 $176 $50 $15 $4 $245 $35 $8 $8 $5 $1 $88 $7 $81 $117 $4 $46 $5 $46 $16 $5 $2 $58 $6 $335 $24 $32 $6 $7 $0 $143 $20 $4 $24 $3 $19 $51 $9 $54 $25 $89 $89 $16 $32 $76 $7 $39 $5 $3 $26 $2 $91 $14 $8 $30 $23 $15 $10 

## 6. Save Model Weights

In [7]:
runner.save("sentence_transformer_model.pth")
print("Saved to sentence_transformer_model.pth")

Saved to sentence_transformer_model.pth


# 1. Sanity check — inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred_original:.2f}")
print(f"Error:   ${abs(pred_original - sample.price):.2f}")
print()

# 2. Load roundtrip test — load lại từ .pth và so sánh kết quả
runner.load("sentence_transformer_model.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch! Before=${pred_original:.2f} After=${pred_loaded:.2f}"
print(f"Load roundtrip test PASSED. Diff: ${diff:.4f}")

In [8]:
# Quick sanity check
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
Actual:  $219.00
Predict: $132.97
Error:   $86.03
